# 🧪 Lab 01: Database Basics

**Topics:** CRUD, projections, operators, bulk writes

---

In [2]:
from pymongo import MongoClient, ReadPreference
from pymongo.read_concern import ReadConcern
from pymongo.write_concern import WriteConcern
from pymongo.errors import ConnectionFailure
from bson import ObjectId
from datetime import datetime, timedelta, timezone
import pprint

USE_ATLAS = False
ATLAS_URI  = "mongodb+srv://<username>:<password>@<cluster>.mongodb.net/?retryWrites=true&w=majority"
DOCKER_URI = "mongodb://127.0.0.1:27017/?directConnection=true"

uri = ATLAS_URI if USE_ATLAS else DOCKER_URI
client = MongoClient(uri, serverSelectionTimeoutMS=5000)
try:
    client.admin.command("ping")
    print("✅ Connected to", "Atlas" if USE_ATLAS else "Docker MongoDB")
except ConnectionFailure as e:
    print("❌ Connection failed:", e)

db = client["mongo_labs"]
pp = pprint.PrettyPrinter(indent=2)

✅ Connected to Docker MongoDB


## CREATE

In [3]:
db.products.drop()
r = db.products.insert_one({"name":"USB Hub","category":"peripherals","price":29.99,"stock":80,"createdAt":datetime.now(timezone.utc)})
print("Inserted:", r.inserted_id)

Inserted: 6a02c8067dbef744dd8f7540


In [4]:
bulk = db.products.insert_many([{"name":"Webcam","category":"peripherals","price":59.99},{"name":"Desk","category":"furniture","price":499.0}])
print("IDs:", bulk.inserted_ids)

IDs: [ObjectId('6a02c8067dbef744dd8f7541'), ObjectId('6a02c8067dbef744dd8f7542')]


## READ

In [5]:
pp.pprint(db.products.find_one({"name":"USB Hub"}))

{ '_id': ObjectId('6a02c8067dbef744dd8f7540'),
  'category': 'peripherals',
  'createdAt': datetime.datetime(2026, 5, 12, 6, 26, 14, 845000),
  'name': 'USB Hub',
  'price': 29.99,
  'stock': 80}


In [6]:
for d in db.products.find({"category":"peripherals"},{"_id":0,"name":1,"price":1}):
    print(f"{d['name']}: ${d['price']}")

USB Hub: $29.99
Webcam: $59.99


In [7]:
# Comparison operators
for d in db.products.find({"price":{"$gte":25,"$lte":100}},{"_id":0,"name":1}):
    print(d["name"])

USB Hub
Webcam


In [8]:
# Logical operators
for d in db.products.find({"$or":[{"category":"peripherals"},{"price":{"$gt":400}}]},{"_id":0,"name":1}):
    print(d["name"])

USB Hub
Webcam
Desk


In [9]:
# sort + limit
for d in db.products.find({},{"_id":0,"name":1,"price":1}).sort("price",-1).limit(3):
    print(d)
print("Count:", db.products.count_documents({"category":"peripherals"}))

{'name': 'Desk', 'price': 499.0}
{'name': 'Webcam', 'price': 59.99}
{'name': 'USB Hub', 'price': 29.99}
Count: 2


## UPDATE

In [10]:
r = db.products.update_one({"name":"USB Hub"},{"$set":{"price":24.99,"onSale":True}})
print("Matched:",r.matched_count,"Modified:",r.modified_count)

Matched: 1 Modified: 1


In [11]:
db.products.update_one({"name":"USB Hub"},{"$inc":{"stock":-5}})
db.products.update_one({"name":"USB Hub"},{"$unset":{"onSale":""}})
print("Updated")

Updated


In [12]:
db.products.update_many({"category":"peripherals"},{"$set":{"updatedAt":datetime.now(timezone.utc)}})
print("Update many done")

Update many done


In [13]:
db.products.update_one({"name":"BT Speaker"},{"$set":{"name":"BT Speaker","category":"audio","price":79.99}},upsert=True)
print("Upserted")

Upserted


In [14]:
db.products.replace_one({"name":"Webcam"},{"name":"Webcam 4K","category":"peripherals","price":89.99})
print("Replaced")

Replaced


## DELETE

In [15]:
print("Deleted:", db.products.delete_one({"name":"BT Speaker"}).deleted_count)
print("Deleted many:", db.products.delete_many({"category":"furniture"}).deleted_count)

Deleted: 1
Deleted many: 1


## BULK WRITE

In [16]:
from pymongo import InsertOne, UpdateOne, DeleteOne
db.products.bulk_write([InsertOne({"name":"Stand","price":35.99}),UpdateOne({"name":"USB Hub"},{"$set":{"price":22.99}}),DeleteOne({"name":"Desk"})],ordered=False)
print("Bulk write done")

Bulk write done
